# factorlib demo — 四種因子類別 × 完整 API 展示

這份 notebook 不是教學文檔，是**可執行的功能索引**：每個 cell 對應 `factorlib/README.md` 的一段 API，照順序跑完就看過 factorlib 全部功能。

涵蓋：
- 四種 `factor_type`：`cross_sectional` / `event_signal` / `macro_panel` / `macro_common`
- 六個使用層級（Level 0–6）：單因子 → 自訂 config → 個別 metrics → batch + BHY → redundancy → charts → MLflow

資料來源全部是 `fl.datasets` 的合成 panel（seeded），從 fresh clone 可以直接跑，**不需要任何外部 parquet**。
如果要改跑自己手上的真實 panel，把 §0.9 那格的 `raw_demo = ...` 換成 `fl.adapt(your_df, ...)` 即可，下游 cells 都建立在 canonical `date / asset_id / price` 之上。

## 0. Setup

In [1]:
from __future__ import annotations

import polars as pl
import factorlib as fl

pl.Config.set_tbl_rows(8)
pl.Config.set_fmt_str_lengths(60)
print('factorlib version:', getattr(fl, '__version__', 'dev'))

factorlib version: 0.1.0


## 0.5 Synthetic quick-start — 不需外部資料

`fl.datasets` 提供兩個 seeded 生成器：
- `make_cs_panel(n_assets, n_dates, ic_target=...)` — CS 面板，每期 factor 與 N 日 forward return 的 CS 相關性 ≈ `ic_target`。
- `make_event_panel(n_assets, n_dates, event_rate, post_event_drift_bps)` — 事件訊號（`factor ∈ {-1, 0, +1}`）加上 post-event drift。

輸出 canonical columns `date, asset_id, price, factor`；下一步照一般流程 `fl.preprocess → fl.evaluate`。
下面這段 cell 示範完整 end-to-end：合成資料 → preprocess → evaluate → profile。後續 sections 會用更大的合成 panel 展示 factor generators、batch、event、macro 等 API。

In [2]:
cfg = fl.CrossSectionalConfig(forward_periods=5)
synthetic_raw = fl.datasets.make_cs_panel(
    n_assets=100, n_dates=500, ic_target=0.04, forward_periods=5, seed=2024,
)
print('synthetic raw:', synthetic_raw.shape, synthetic_raw.columns)

synthetic_prepared = fl.preprocess(synthetic_raw, config=cfg)
synthetic_profile = fl.evaluate(synthetic_prepared, 'synthetic', config=cfg)
print(f'realized ic_mean = {synthetic_profile.ic_mean:.4f} '
      f'(target was 0.04)')
print('verdict:', synthetic_profile.verdict())

synthetic raw: (50000, 4) ['date', 'asset_id', 'price', 'factor']
realized ic_mean = 0.0352 (target was 0.04)
verdict: FAILED


## 0.9 共用 price panel（給 §1 之後的所有 cells）

用 `fl.datasets.make_cs_panel` 產一份 100 資產 × 500 日的合成 price panel，只留 canonical `date / asset_id / price` — factor 欄位在這裡丟掉，後續 sections 會用 `factorlib.factors` 的 generator 從價格重算動量/波動度等真正的候選因子。
若要換成真實資料，只要把這格的 `raw_demo = ...` 改成 `fl.adapt(your_df, date=..., asset_id=..., price=...)` 即可。

In [3]:
raw_demo = (
    fl.datasets.make_cs_panel(n_assets=100, n_dates=500, seed=2024)
    .select(['date', 'asset_id', 'price'])
)
print('rows:', raw_demo.height, '| assets:', raw_demo['asset_id'].n_unique(),
      '| span:', raw_demo['date'].min(), '->', raw_demo['date'].max())
raw_demo.head(3)

rows: 50000 | assets: 100 | span: 2020-01-02 00:00:00 -> 2021-05-15 00:00:00


date,asset_id,price
datetime[ms],str,f64
2020-01-02 00:00:00,"""A0000""",101.087141
2020-01-02 00:00:00,"""A0001""",99.915825
2020-01-02 00:00:00,"""A0002""",98.631684


## 1. 四種因子類別總覽

`fl.describe_factor_types()` 印出所有註冊的 factor_type 與用途；
`fl.describe_profile(<type>)` 反射對應 Profile dataclass 的欄位、canonical p 與方法。
這兩個是「不用開檔就知道 factorlib 現在長什麼樣」的入口。

In [4]:
fl.describe_factor_types()
print()
print('list_factor_types ->', fl.list_factor_types())

  cross_sectional     : 截面因子（每期每資產有 signal，N ≥ 30）→ 選股
  event_signal        : 事件訊號（離散觸發）→ 事件交易
  macro_panel         : 宏觀 panel（小截面 N < 30）→ 跨國配置
  macro_common        : 宏觀共用（單一時序）→ 風險歸因

list_factor_types -> ['cross_sectional', 'event_signal', 'macro_panel', 'macro_common']


In [5]:
for ft in fl.list_factor_types():
    fl.describe_profile(ft)


  cross_sectional — CrossSectionalProfile
  ──────────────────────────────────────────────────
  Typed profile for a cross-sectional factor.

  Fields:
    factor_name                   str
    n_periods                     int
    ic_mean                       float
    ic_tstat                      float
    ic_p                          float  [PValue]
    ic_ir                         float
    hit_rate                      float
    hit_rate_p                    float  [PValue]
    ic_trend                      float
    ic_trend_p                    float  [PValue]
    monotonicity                  float
    quantile_spread               float
    spread_tstat                  float
    spread_p                      float  [PValue]
    top_concentration             float
    top_concentration_eff_ratio   float
    oos_survival_ratio            float
    oos_sign_flipped              bool
    median_universe_n             int
    orthogonalize_r2_mean         float | None
    ort

## 2. Cross-sectional（選股因子）

訊號型態：每期每資產有連續值。典型用法 = 動量 / 價值 / 規模。
Canonical test: `ic_p`（IC 非重疊 t-test）。

### 2.1 Build factor（用 factorlib 內建 generator）

In [6]:
from factorlib.factors import generate_momentum, generate_momentum_60d
from factorlib.factors.volatility import generate_volatility

mom20 = generate_momentum(raw_demo, lookback=20)
print('columns:', mom20.columns)
print('shape:', mom20.shape)
mom20.select(['asset_id', 'date', 'price', 'factor']).head(3)

columns: ['date', 'asset_id', 'price', 'factor']
shape: (48000, 4)


asset_id,date,price,factor
str,datetime[ms],f64,f64
"""A0000""",2020-01-22 00:00:00,111.528983,0.103295
"""A0000""",2020-01-23 00:00:00,112.877851,0.131887
"""A0000""",2020-01-24 00:00:00,109.010596,0.103993


### 2.2 Level 0 — 最簡單 `fl.evaluate`

`fl.preprocess` 與 `fl.evaluate` 是兩步驟：preprocess 產出含 `forward_return` 等中間欄位的 prepared panel，evaluate 再跑 all metrics + diagnostics，回傳 `CrossSectionalProfile`（frozen dataclass）。把 preprocess 留在 user code 是為了避免 config 兩邊不一致偷偷壞掉 canonical_p。

In [7]:
cfg_default = fl.CrossSectionalConfig()
mom20_prep = fl.preprocess(mom20, config=cfg_default)
profile = fl.evaluate(mom20_prep, 'Mom_20D', config=cfg_default)

print('type:       ', type(profile).__name__)
print('verdict:    ', profile.verdict())
print('canonical_p:', f'{profile.canonical_p:.4f}')
print('ic_mean:    ', f'{profile.ic_mean:+.4f}')
print('ic_tstat:   ', f'{profile.ic_tstat:+.2f}')
print('ic_ir:      ', f'{profile.ic_ir:+.3f}')
print('long-short spread:', f'{profile.quantile_spread:+.4f}')
print('turnover:   ', f'{profile.turnover:.2%}')
print('net_spread: ', f'{profile.net_spread:+.4f}')

type:        CrossSectionalProfile
verdict:     FAILED
canonical_p: 0.0996
ic_mean:     -0.0087
ic_tstat:    -1.66
ic_ir:       -0.082
long-short spread: -0.0008
turnover:    6.26%
net_spread:  -0.0012


In [8]:
# profile.diagnose() 回傳結構化 Diagnostic — 包含 severity / code / message
for d in profile.diagnose():
    print(f'[{d.severity:<7s}] {d.code:<35s} {d.message}')

[warn   ] cs.small_universe                   Median universe size < 200 — quantile analysis may be unstable; consider MacroPanelConfig (smaller-N workflows).
[veto   ] cs.oos_sign_flipped                 OOS sample shows a sign flip vs IS — overfitting risk; the factor did not generalize.


### 2.3 Level 1 — 自訂 `CrossSectionalConfig`

切換 forward horizon、quantile groups、trading-cost 估計等。

In [9]:
cfg = fl.CrossSectionalConfig(
    forward_periods=10,            # 10 日 forward return
    n_groups=5,                    # 5 組 quantile
    mad_n=3.0,
    return_clip_pct=(0.01, 0.99),
    estimated_cost_bps=30,
)
mom20_prep_10d = fl.preprocess(mom20, config=cfg)
profile_10d = fl.evaluate(mom20_prep_10d, 'Mom_20D_h10', config=cfg)
print('verdict @10d:', profile_10d.verdict(), '| ic_ir:', f'{profile_10d.ic_ir:+.3f}')

verdict @10d: FAILED | ic_ir: -0.042


### 2.4 Level 2 — 個別 metrics（繞過 Profile）

當你只想算某個統計量、或要餵 metric 進自己的 pipeline，不需要整個 Profile。
流程：`fl.preprocess` → 個別 `metrics.*` 函式（全部回 `MetricOutput`）。

In [10]:
from factorlib.metrics import (
    compute_ic, ic, ic_ir, quantile_spread, monotonicity,
)

prepared = fl.preprocess(mom20, config=cfg)
print('prepared columns:', prepared.columns, '\n')

ic_series = compute_ic(prepared)  # DataFrame[date, ic]
ic_m = ic(ic_series, forward_periods=cfg.forward_periods)
ir_m = ic_ir(ic_series)
spread_m = quantile_spread(prepared, forward_periods=cfg.forward_periods, n_groups=cfg.n_groups)
mono_m = monotonicity(prepared, forward_periods=cfg.forward_periods, n_groups=cfg.n_groups)

# MetricOutput 的 p-value 放在 .metadata['p_value']，__repr__ 會印摘要
for name, out in [('ic', ic_m), ('ic_ir', ir_m), ('quantile_spread', spread_m), ('monotonicity', mono_m)]:
    p = out.metadata.get('p_value')
    p_str = 'n/a' if p is None else f'{p:.4f}'
    stat_str = 'n/a' if out.stat is None else f'{out.stat:+.2f}'
    print(f'{name:<15s} value={out.value:+.4f}  stat={stat_str:>6s}  p={p_str}  sig={out.significance or ""}')

prepared columns: ['date', 'asset_id', 'factor_raw', 'factor', 'forward_return', 'abnormal_return', 'price', '_fl_forward_periods'] 

ic              value=-0.0045  stat= -0.66  p=0.5121  sig=
ic_ir           value=-0.0423  stat=   n/a  p=n/a  sig=
quantile_spread value=-0.0005  stat= -1.62  p=0.1113  sig=
monotonicity    value=+0.4787  stat= -1.09  p=0.2830  sig=


### 2.5 Level 3 — `evaluate_batch` + BHY + rank + top

批次評估多因子，回傳 polars-native `ProfileSet`。
`multiple_testing_correct` 做 BHY 多重檢定校正；`filter` / `rank_by` / `top` 可鏈式串接。

In [11]:
# 準備三個候選 CS 因子
mom60 = generate_momentum_60d(raw_demo)
vol20 = generate_volatility(raw_demo, lookback=20)

factors_map = {
    'Mom_20D': mom20,
    'Mom_60_5': mom60,
    'Vol_20D': vol20,
}
cs_cfg = fl.CrossSectionalConfig()
prepared_map = {name: fl.preprocess(df, config=cs_cfg) for name, df in factors_map.items()}
ps = fl.evaluate_batch(prepared_map, config=cs_cfg)
print('ProfileSet size:', len(ps), '| profile class:', ps.profile_cls.__name__)
ps.to_polars().select(['factor_name', 'ic_mean', 'ic_ir', 'quantile_spread', 'canonical_p'])  # quick glance


ProfileSet size: 3 | profile class: CrossSectionalProfile


factor_name,ic_mean,ic_ir,quantile_spread,canonical_p
str,f64,f64,f64,f64
"""Mom_20D""",-0.008722,-0.082259,-0.000786,0.099556
"""Mom_60_5""",-0.001873,-0.017061,-0.000262,0.680864
"""Vol_20D""",0.013936,0.132268,-0.00025,0.170158


In [12]:
# BHY 多重檢定 + 按 IC_IR 排序 + 取 top 2
top = (
    ps
    .multiple_testing_correct(p_source='canonical_p', fdr=0.10)
    .rank_by('ic_ir', descending=True)
    .top(2)
)
top.to_polars().select([
    'factor_name', 'ic_p', 'p_adjusted', 'bhy_significant', 'ic_ir',
])

factor_name,ic_p,p_adjusted,bhy_significant,ic_ir
str,f64,f64,bool,f64
"""Vol_20D""",0.170158,0.467934,false,0.132268
"""Mom_60_5""",0.680864,1.0,false,-0.017061


In [13]:
# 也能傳 pl.Expr 或 Callable[[Profile], bool] 給 filter
stable = ps.filter(pl.col('ic_ir').abs() > 0.1)
print('factors with |IC_IR| > 0.1:', [p.factor_name for p in stable])

factors with |IC_IR| > 0.1: ['Vol_20D']


### 2.6 Level 4 — Redundancy matrix

多因子之間的成對 `|ρ|`，抓出冗餘訊號。

In [14]:
# redundancy_matrix 需要 per-factor Artifacts；用 keep_artifacts=True 保留。
ps2, arts = fl.evaluate_batch(
    prepared_map, config=cs_cfg, keep_artifacts=True,
)

redund = fl.redundancy_matrix(ps2, method='value_series', artifacts=arts)
redund

/Users/cfh00884862/Desktop/dst/code/factor-analysis/external/factorlib/factorlib/metrics/redundancy.py:114: UserWarning: redundancy_matrix: factors ['Mom_60_5'] have missing dates that other factors cover; correlation is computed on the intersection. If that shrinks the window unacceptably, drop the short-history factors from the ProfileSet.
  return _value_series_matrix(names, artifacts, profiles.profile_cls)


factor,Mom_20D,Mom_60_5,Vol_20D
str,f64,f64,f64
"""Mom_20D""",1.0,0.415135,0.035454
"""Mom_60_5""",0.415135,1.0,0.080456
"""Vol_20D""",0.035454,0.080456,1.0


### 2.7 Level 5 — Charts (optional dep: `factorlib[charts]`)

`build_artifacts` 保留中間結果（IC series / spread series / quantile group returns…），
丟進 `report_charts` 產 plotly 圖。未安裝 `plotly` 會 raise ImportError — 此處 try/except 包起來。

In [15]:
from factorlib.evaluation.pipeline import build_artifacts

prepared = fl.preprocess(mom20, config=fl.CrossSectionalConfig())
artifacts = build_artifacts(prepared, fl.CrossSectionalConfig())
artifacts.factor_name = 'Mom_20D'
print('artifacts.intermediates keys:', sorted(artifacts.intermediates.keys()))

try:
    from factorlib.charts import report_charts
    figs = report_charts(artifacts)
    print(f'produced {len(figs)} figures:', list(figs)[:5])
    # 在 notebook 直接顯示第一張
    next(iter(figs.values())).show()
except ImportError as e:
    print('charts skipped — install with `pip install factorlib[charts]`:', e)

artifacts.intermediates keys: ['ic_series', 'ic_values', 'spread_series']


produced 4 figures: ['cumulative_ic', 'ic_distribution', 'spread_ts', 'quantile_returns']


### 2.8 Level 6 — MLflow tracking (optional dep: `factorlib[mlflow]`)

`on_result` callback 在 `evaluate_batch` 的每個因子算完後觸發，
可用來 log profile 到 MLflow experiment。這裡只示範寫法，不實跑（避免副作用）。

In [16]:
# 不實跑 — 只展示 API 形狀
demo = '''
from factorlib.integrations.mlflow import FactorTracker

tracker = FactorTracker('Factor_Zoo')
ps = fl.evaluate_batch(
    factors_map,
    factor_type='cross_sectional',
    on_result=lambda name, p: tracker.log_profile(p, factor_type='cross_sectional'),
)
'''
print(demo)


from factorlib.integrations.mlflow import FactorTracker

tracker = FactorTracker('Factor_Zoo')
ps = fl.evaluate_batch(
    factors_map,
    factor_type='cross_sectional',
    on_result=lambda name, p: tracker.log_profile(p, factor_type='cross_sectional'),
)



## 3. Event-signal（事件交易因子）

訊號型態：離散觸發 `{-1, 0, +1}`。Canonical test: `caar_p`（CAAR 非重疊 t-test）。
範例：黃金交叉 / 死亡交叉 → 事件後 5 日有異常報酬嗎？

### 3.1 Build golden/death-cross event signal

In [17]:
event_df = (
    raw_demo.sort(['asset_id', 'date'])
    .with_columns(
        pl.col('price').rolling_mean(5).over('asset_id').alias('ma5'),
        pl.col('price').rolling_mean(20).over('asset_id').alias('ma20'),
    )
    .with_columns(
        pl.when(pl.col('ma5') > pl.col('ma20')).then(1)
          .otherwise(-1).alias('cross_state')
    )
    .with_columns(
        (pl.col('cross_state') - pl.col('cross_state').shift(1).over('asset_id')).alias('delta')
    )
    .with_columns(
        pl.when(pl.col('delta') == 2).then(1.0)   # 黃金交叉
        .when(pl.col('delta') == -2).then(-1.0)   # 死亡交叉
        .otherwise(0.0).alias('factor')
    )
    .filter(pl.col('factor').is_not_null() & pl.col('ma20').is_not_null())
    .select(['asset_id', 'date', 'price', 'factor'])
)
print('factor value counts:')
print(event_df['factor'].value_counts().sort('factor'))

factor value counts:
shape: (3, 2)
┌────────┬───────┐
│ factor ┆ count │
│ ---    ┆ ---   │
│ f64    ┆ u32   │
╞════════╪═══════╡
│ -1.0   ┆ 1526  │
│ 0.0    ┆ 45004 │
│ 1.0    ┆ 1570  │
└────────┴───────┘


### 3.2 evaluate（自動切到 `EventProfile`）

In [18]:
ev_cfg = fl.EventConfig(
    forward_periods=5,
    event_window_pre=5,
    event_window_post=20,
    cluster_window=3,
    adjust_clustering='none',
)
event_prepared = fl.preprocess(event_df, config=ev_cfg)
ev_profile = fl.evaluate(event_prepared, 'GoldenCross', config=ev_cfg)

print('type:           ', type(ev_profile).__name__)
print('n_events:       ', ev_profile.n_events)
print('n_periods (dates):', ev_profile.n_periods)
print('verdict:        ', ev_profile.verdict())
print('CAAR mean:      ', f'{ev_profile.caar_mean:+.4f}')
print('CAAR p (canonical):', f'{ev_profile.caar_p:.4f}')
print('BMP z:          ', f'{ev_profile.bmp_zstat:+.2f}  p={ev_profile.bmp_p:.4f}')
print('event_hit_rate: ', f'{ev_profile.event_hit_rate:.2%}  p={ev_profile.event_hit_rate_p:.4f}')
print('profit_factor:  ', f'{ev_profile.profit_factor:.3f}')
print('clustering_hhi (normalized):', ev_profile.clustering_hhi_normalized)

type:            EventProfile
n_events:        3049
n_periods (dates): 475
verdict:         FAILED
CAAR mean:       +0.0000
CAAR p (canonical): 0.9180
BMP z:           +0.25  p=0.8056
event_hit_rate:  49.85%  p=0.8705
profit_factor:   0.998
clustering_hhi (normalized): 0.00041367898592117755


In [19]:
for d in ev_profile.diagnose():
    print(f'[{d.severity:<7s}] {d.code:<35s} {d.message}')

[veto   ] event.oos_sign_flipped              OOS sample shows a sign flip vs IS — overfitting risk.


### 3.3 Level 2 — individual event metrics

In [20]:
from factorlib.metrics import (
    compute_caar, caar, bmp_test, event_hit_rate,
    corrado_rank_test, clustering_diagnostic,
)

ev_prepared = fl.preprocess(event_df, config=ev_cfg)

caar_series = compute_caar(ev_prepared)   # per-event-date signed AR
print('caar:         ', caar(caar_series, forward_periods=5))
print('BMP:          ', bmp_test(ev_prepared, forward_periods=5))
print('event_hit_rate:', event_hit_rate(ev_prepared))
print('corrado rank: ', corrado_rank_test(ev_prepared))
print('clustering:   ', clustering_diagnostic(ev_prepared, cluster_window=3))

caar:          MetricOutput(caar=0.0000, stat=0.01)
BMP:           MetricOutput(bmp_test=0.0046, stat=0.24)
event_hit_rate: MetricOutput(event_hit_rate=0.4995, stat=-0.05)
corrado rank:  MetricOutput(corrado_rank=0.0019, stat=0.36)
clustering:    MetricOutput(clustering_hhi=0.0025)


## 4. Macro panel（跨國/小截面配置因子）

訊號型態：連續值 + 小截面 `N < 30`。典型用法 = 跨國 CPI / 利差配置。
Canonical test: `fm_beta_p`（Fama-MacBeth Newey-West t-test）。

這裡另外生一份 25 資產的合成 panel，當「25 個 pseudo-country」，示範小截面情境下的 `MacroPanelConfig`。

### 4.1 Build small-N panel

In [21]:
small_raw = (
    fl.datasets.make_cs_panel(n_assets=25, n_dates=500, seed=7)
    .select(['date', 'asset_id', 'price'])
)
# factor = 相對 60 日均價偏離（mean-reversion proxy）
small_panel = small_raw.with_columns(
    (pl.col('price') / pl.col('price').rolling_mean(60).over('asset_id') - 1).alias('factor')
).drop_nulls('factor')
print('small-N panel:', small_panel['asset_id'].n_unique(), 'assets |', small_panel.height, 'rows')
small_panel.head(3)

small-N panel: 25 assets | 11025 rows


date,asset_id,price,factor
datetime[ms],str,f64,f64
2020-03-01 00:00:00,"""A0000""",84.231739,-0.084472
2020-03-01 00:00:00,"""A0001""",120.529708,0.295908
2020-03-01 00:00:00,"""A0002""",66.084363,-0.223578


### 4.2 evaluate + `MacroPanelConfig`

In [22]:
mp_cfg = fl.MacroPanelConfig(
    forward_periods=5,
    n_groups=3,               # 小截面 → 少分組
    demean_cross_section=False,
    min_cross_section=10,
)
small_prepared = fl.preprocess(small_panel, config=mp_cfg)
mp_profile = fl.evaluate(small_prepared, 'SmallRelValue', config=mp_cfg)

print('verdict:              ', mp_profile.verdict())
print('fm_beta_mean (λ):     ', f'{mp_profile.fm_beta_mean:+.5f}')
print('fm_beta_tstat:        ', f'{mp_profile.fm_beta_tstat:+.2f}')
print('fm_beta_p (canonical):', f'{mp_profile.fm_beta_p:.4f}')
print('pooled_beta:          ', f'{mp_profile.pooled_beta:+.5f}  p={mp_profile.pooled_beta_p:.4f}')
print('beta_sign_consistency:', f'{mp_profile.beta_sign_consistency:.2%}')
print('quantile_spread:         ', f'{mp_profile.quantile_spread:+.4f}')
print('median cross-section N:', mp_profile.median_cross_section_n)

verdict:               FAILED
fm_beta_mean (λ):      +0.00015
fm_beta_tstat:         +0.79
fm_beta_p (canonical): 0.4304
pooled_beta:           +0.00022  p=0.0080
beta_sign_consistency: 56.32%
quantile_spread:          -0.0001
median cross-section N: 25


## 5. Macro common（全市場共用因子）

訊號型態：單一時序，每個資產共用同一個 factor 值。典型用法 = VIX / 黃金 / USD index。
Canonical test: `ts_beta_p`（per-asset 時序 OLS β 的截面 t-test）。

範例：市場截面波動率（所有資產日報酬的 cross-sectional std）→ 對每檔股票的 exposure 穩定嗎？

### 5.1 Build market vol + broadcast 到每個資產

In [23]:
# 市場波動率（每日：跨資產日報酬 std）
mkt_vol = (
    raw_demo.sort(['asset_id', 'date'])
    .with_columns(pl.col('price').pct_change().over('asset_id').alias('ret'))
    .group_by('date').agg(pl.col('ret').std().alias('factor'))
    .sort('date').drop_nulls('factor')
)
print('market vol head:')
print(mkt_vol.head(3))

# 為了控 runtime，sample 50 檔標的
sample_assets = raw_demo.select('asset_id').unique().head(50)['asset_id'].to_list()
common_df = (
    raw_demo.filter(pl.col('asset_id').is_in(sample_assets))
    .select(['date', 'asset_id', 'price'])
    .join(mkt_vol, on='date', how='inner')
)
print('common panel shape:', common_df.shape)

market vol head:
shape: (3, 2)
┌─────────────────────┬──────────┐
│ date                ┆ factor   │
│ ---                 ┆ ---      │
│ datetime[ms]        ┆ f64      │
╞═════════════════════╪══════════╡
│ 2020-01-03 00:00:00 ┆ 0.019864 │
│ 2020-01-04 00:00:00 ┆ 0.019671 │
│ 2020-01-05 00:00:00 ┆ 0.017277 │
└─────────────────────┴──────────┘
common panel shape: (24950, 4)


### 5.2 evaluate + `MacroCommonConfig`

In [24]:
mc_cfg = fl.MacroCommonConfig(
    forward_periods=5,
    ts_window=60,
    tradable=False,
)
common_prepared = fl.preprocess(common_df, config=mc_cfg)
mc_profile = fl.evaluate(common_prepared, 'MktVol', config=mc_cfg)

print('verdict:              ', mc_profile.verdict())
print('n_assets:             ', mc_profile.n_assets)
print('ts_beta_mean:         ', f'{mc_profile.ts_beta_mean:+.5f}')
print('ts_beta_tstat:        ', f'{mc_profile.ts_beta_tstat:+.2f}')
print('ts_beta_p (canonical):', f'{mc_profile.ts_beta_p:.4f}')
print('mean R²:              ', f'{mc_profile.mean_r_squared:.4f}')
print('beta_sign_consistency:', f'{mc_profile.ts_beta_sign_consistency:.2%}')

verdict:               PASS
n_assets:              50
ts_beta_mean:          -0.00014
ts_beta_tstat:         -2.16
ts_beta_p (canonical): 0.0357
mean R²:               0.0029
beta_sign_consistency: 64.00%


## 6. 切換因子類別的 cheat sheet

只要換 `factor_type=` 字串（或對應的 `XxxConfig`），同一組 `fl.evaluate` / `fl.evaluate_batch` / `ProfileSet` API 都能用——差別在回傳的 Profile dataclass 欄位不同、canonical p 對應的統計量不同。

In [25]:
cheat = pl.DataFrame({
    'factor_type': ['cross_sectional', 'event_signal', 'macro_panel', 'macro_common'],
    'signal_shape': ['每期每資產連續值', '離散 {-1,0,+1}', '連續值，小截面 N<30', '單一時序、全資產共用'],
    'Config':       ['CrossSectionalConfig', 'EventConfig', 'MacroPanelConfig', 'MacroCommonConfig'],
    'Profile':      ['CrossSectionalProfile', 'EventProfile', 'MacroPanelProfile', 'MacroCommonProfile'],
    'canonical_p':  ['ic_p', 'caar_p', 'fm_beta_p', 'ts_beta_p'],
    'core_question': ['排序能預測截面報酬差異嗎？', '事件後報酬有異常嗎？',
                       '宏觀指標能預測跨資產配置嗎？', '資產對共同因子的 exposure 穩定嗎？'],
})
cheat

factor_type,signal_shape,Config,Profile,canonical_p,core_question
str,str,str,str,str,str
"""cross_sectional""","""每期每資產連續值""","""CrossSectionalConfig""","""CrossSectionalProfile""","""ic_p""","""排序能預測截面報酬差異嗎？"""
"""event_signal""","""離散 {-1,0,+1}""","""EventConfig""","""EventProfile""","""caar_p""","""事件後報酬有異常嗎？"""
"""macro_panel""","""連續值，小截面 N<30""","""MacroPanelConfig""","""MacroPanelProfile""","""fm_beta_p""","""宏觀指標能預測跨資產配置嗎？"""
"""macro_common""","""單一時序、全資產共用""","""MacroCommonConfig""","""MacroCommonProfile""","""ts_beta_p""","""資產對共同因子的 exposure 穩定嗎？"""


---

## 附錄 A：全 factor_type 收到的 Profile 一覽

把本 notebook 跑出來的四個 profile 並排，展示不同 factor_type 的 verdict + canonical_p 是同一套介面。

In [26]:
summary = pl.DataFrame([
    {
        'factor_type': 'cross_sectional',
        'factor_name': profile.factor_name,
        'verdict': profile.verdict(),
        'canonical_p': profile.canonical_p,
        'n_periods': profile.n_periods,
    },
    {
        'factor_type': 'event_signal',
        'factor_name': ev_profile.factor_name,
        'verdict': ev_profile.verdict(),
        'canonical_p': ev_profile.canonical_p,
        'n_periods': ev_profile.n_periods,
    },
    {
        'factor_type': 'macro_panel',
        'factor_name': mp_profile.factor_name,
        'verdict': mp_profile.verdict(),
        'canonical_p': mp_profile.canonical_p,
        'n_periods': mp_profile.n_periods,
    },
    {
        'factor_type': 'macro_common',
        'factor_name': mc_profile.factor_name,
        'verdict': mc_profile.verdict(),
        'canonical_p': mc_profile.canonical_p,
        'n_periods': mc_profile.n_periods,
    },
])
summary

factor_type,factor_name,verdict,canonical_p,n_periods
str,str,str,f64,i64
"""cross_sectional""","""Mom_20D""","""FAILED""",0.099556,474
"""event_signal""","""GoldenCross""","""FAILED""",0.918012,475
"""macro_panel""","""SmallRelValue""","""FAILED""",0.430356,435
"""macro_common""","""MktVol""","""PASS""",0.035717,493
